In [1]:
import instaloader
import json
import os
from datetime import datetime
import time
from pathlib import Path

In [2]:
class InstagramPostDownloader:
    """
    Classe para baixar imagens e metadados de posts do Instagram.
    """
    
    def __init__(self, pasta_base):
        """
        Inicializa o downloader.
        
        Args:
            pasta_base (str): Pasta base para salvar os downloads
        """
        self.pasta_base = pasta_base
        self.loader = instaloader.Instaloader(
            download_videos=False,
            download_video_thumbnails=False,
            download_geotags=False,
            download_comments=False,  # Comentários serão salvos no JSON
            save_metadata=True,
            compress_json=False,
            max_connection_attempts=3,
            sleep=True,
            quiet=True
        )
        
    def fazer_login(self, username=None, password=None):
        """
        Faz login no Instagram (opcional, mas recomendado).
        
        Args:
            username (str): Nome de usuário
            password (str): Senha
        """
        try:
            if username and password:
                self.loader.login(username, password)
                print("✅ Login realizado com sucesso!")
            else:
                print("⚠️  Continuando sem login (pode haver limitações)")
        except Exception as e:
            print(f"❌ Erro no login: {e}")
    
    def processar_post(self, post, pasta_post, indice):
        """
        Processa um post individual: baixa a imagem e cria JSON com metadados.
        
        Args:
            post: Objeto Post do Instaloader
            pasta_post (str): Pasta onde salvar os arquivos deste post
            indice (int): Índice do post para nomear os arquivos
            
        Returns:
            dict: Informações processadas do post
        """
        # Cria pasta para o post
        Path(pasta_post).mkdir(parents=True, exist_ok=True)
        
        # Configura o loader para salvar na pasta correta
        self.loader.dirname_pattern = pasta_post
        self.loader.filename_pattern = f"post_{indice:03d}"
        
        # Baixa a imagem usando o Instaloader
        self.loader.download_post(post, target=pasta_post)
        
        # Renomeia os arquivos para um padrão consistente
        for arquivo in os.listdir(pasta_post):
            if arquivo.startswith(f"{post.shortcode}"):
                extensao = os.path.splitext(arquivo)[1]
                novo_nome = f"post_{indice:03d}_{post.shortcode}{extensao}"
                caminho_antigo = os.path.join(pasta_post, arquivo)
                caminho_novo = os.path.join(pasta_post, novo_nome)
                
                # Evita renomear se já estiver com o nome correto
                if arquivo != novo_nome and not os.path.exists(caminho_novo):
                    os.rename(caminho_antigo, caminho_novo)
        
        # Estrutura de dados completa do post
        post_data = {
            "post_id": post.shortcode,
            "indice": indice,
            "url": f"https://www.instagram.com/p/{post.shortcode}/",
            "tipo": post.typename,
            "data_postagem": post.date.isoformat(),
            "data_postagem_formatada": post.date.strftime("%d/%m/%Y %H:%M"),
            "curtidas": post.likes,
            "comentarios": post.comments,
            "legenda": post.caption if post.caption else "",
            "legenda_completa": post.caption if post.caption else "",
            "visualizacoes": None,
            
            # Informações do perfil
            "perfil": {
                "username": post.owner_username,
                "nome_completo": post.owner_profile.full_name if post.owner_profile else "",
                "seguidores": post.owner_profile.followers if post.owner_profile else None,
                "id": post.owner_id
            },
            
            # Engajamento
            "engajamento": {
                "taxa_curtidas": None,
                "taxa_comentarios": None
            },
            
            # Mídia
            "midia": {
                "url_imagem": None,
                "url_video": None,
                "thumbnail": None,
                "multiplas_midias": []
            },
            
            # Localização (se disponível)
            "localizacao": {
                "nome": post.location.name if post.location else None,
                "id": post.location.id if post.location else None
            },
            
            # Hashtags e menções
            "hashtags": [],
            "mencoes": [],
            
            # Estatísticas adicionais
            "estatisticas": {
                "video_views": post.video_view_count if hasattr(post, 'video_view_count') else None,
                "video_duration": post.video_duration if hasattr(post, 'video_duration') else None,
                "is_video": post.is_video,
                "is_pinned": post.is_pinned if hasattr(post, 'is_pinned') else False
            },
            
            # Metadados técnicos
            "tecnicos": {
                "arquivo_imagem": f"post_{indice:03d}_{post.shortcode}.jpg",
                "tamanho_imagens": []
            }
        }
        
        # Processa URLs das imagens/vídeos
        if post.typename == 'GraphImage':
            post_data["midia"]["url_imagem"] = post.url
            post_data["midia"]["multiplas_midias"].append({
                "tipo": "imagem",
                "url": post.url,
                "indice": 1
            })
            
        elif post.typename == 'GraphVideo':
            post_data["midia"]["url_video"] = post.video_url
            post_data["midia"]["thumbnail"] = post.url
            post_data["midia"]["multiplas_midias"].append({
                "tipo": "video",
                "url": post.video_url,
                "thumbnail": post.url,
                "views": post.video_view_count,
                "duracao": post.video_duration,
                "indice": 1
            })
            
        elif post.typename == 'GraphSidecar':
            # Post com múltiplas mídias (carrossel)
            for idx, node in enumerate(post.get_sidecar_nodes()):
                if node.is_video:
                    post_data["midia"]["multiplas_midias"].append({
                        "tipo": "video",
                        "url": node.video_url,
                        "thumbnail": node.display_url,
                        "views": node.video_view_count if hasattr(node, 'video_view_count') else None,
                        "indice": idx + 1
                    })
                else:
                    post_data["midia"]["multiplas_midias"].append({
                        "tipo": "imagem",
                        "url": node.display_url,
                        "indice": idx + 1
                    })
        
        # Extrai hashtags da legenda
        if post.caption:
            import re
            post_data["hashtags"] = list(set(re.findall(r'#(\w+)', post.caption)))
            post_data["mencoes"] = list(set(re.findall(r'@(\w+)', post.caption)))
        
        # Calcula taxa de engajamento
        if post.owner_profile and post.owner_profile.followers:
            followers = post.owner_profile.followers
            if followers > 0:
                post_data["engajamento"]["taxa_curtidas"] = round((post.likes / followers) * 100, 2)
                post_data["engajamento"]["taxa_comentarios"] = round((post.comments / followers) * 100, 4)
        
        # Verifica o tamanho da imagem baixada
        caminho_imagem = os.path.join(pasta_post, f"post_{indice:03d}_{post.shortcode}.jpg")
        if os.path.exists(caminho_imagem):
            tamanho = os.path.getsize(caminho_imagem)
            post_data["tecnicos"]["tamanho_imagens"].append({
                "arquivo": f"post_{indice:03d}_{post.shortcode}.jpg",
                "tamanho_bytes": tamanho,
                "tamanho_kb": round(tamanho / 1024, 2)
            })
        
        return post_data
    
    def baixar_posts(self, username, num_posts=20, incluir_comentarios=False):
        """
        Baixa posts de um perfil com imagens e JSON estruturado.
        
        Args:
            username (str): Nome de usuário do perfil
            num_posts (int): Número de posts para baixar
            incluir_comentarios (bool): Se deve incluir comentários (pode ser lento)
            
        Returns:
            dict: Resumo do download
        """
        
        print(f"\n{'='*60}")
        print(f"📸 Iniciando download do perfil: @{username}")
        print(f"{'='*60}")
        
        try:
            # Carrega o perfil
            profile = instaloader.Profile.from_username(self.loader.context, username)
            
            # Informações do perfil
            profile_data = {
                "username": profile.username,
                "nome_completo": profile.full_name,
                "biografia": profile.biography,
                "seguidores": profile.followers,
                "seguindo": profile.followees,
                "total_posts": profile.mediacount,
                "conta_privada": profile.is_private,
                "conta_verificada": profile.is_verified,
                "url_externa": profile.external_url,
                "business_category": profile.business_category_name if hasattr(profile, 'business_category_name') else None,
                "data_coleta": datetime.now().isoformat()
            }
            
            print(f"\n📊 Perfil: {profile.full_name}")
            print(f"   Seguidores: {profile.followers:,}")
            print(f"   Posts totais: {profile.mediacount}")
            print(f"   Baixando últimos {num_posts} posts...")
            
            # Cria pasta principal para o perfil (com timestamp)
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            pasta_principal = os.path.join(
                self.pasta_base, 
                f"{username}_{timestamp}"
            )
            Path(pasta_principal).mkdir(parents=True, exist_ok=True)
            
            # Salva informações do perfil
            with open(os.path.join(pasta_principal, "perfil.json"), 'w', encoding='utf-8') as f:
                json.dump(profile_data, f, ensure_ascii=False, indent=2)
            
            # Lista para armazenar todos os posts
            todos_posts = []
            posts_baixados = 0
            
            # Itera sobre os posts
            for i, post in enumerate(profile.get_posts()):
                if i >= num_posts:
                    break
                
                print(f"\n📥 Post {i+1}/{num_posts} (ID: {post.shortcode})")
                
                # Pasta para este post - formato post_001, post_002, etc.
                pasta_post = os.path.join(pasta_principal, f"post_{i+1:03d}")
                
                # Processa o post
                post_data = self.processar_post(post, pasta_post, i+1)
                
                # Se solicitado, adiciona comentários
                if incluir_comentarios:
                    try:
                        comentarios = []
                        for comment in post.get_comments():
                            comentarios.append({
                                "usuario": comment.owner.username,
                                "texto": comment.text,
                                "curtidas": comment.likes_count,
                                "data": comment.created_at_utc.isoformat() if comment.created_at_utc else None
                            })
                        post_data["comentarios_lista"] = comentarios
                        post_data["total_comentarios"] = len(comentarios)
                        print(f"   💬 Comentários incluídos: {len(comentarios)}")
                    except Exception as e:
                        print(f"   ⚠️ Erro ao buscar comentários: {e}")
                
                # Salva JSON do post dentro da pasta do post
                json_path = os.path.join(pasta_post, f"post_{i+1:03d}_{post.shortcode}.json")
                with open(json_path, 'w', encoding='utf-8') as f:
                    json.dump(post_data, f, ensure_ascii=False, indent=2)
                
                # Adiciona à lista geral
                todos_posts.append({
                    "indice": i+1,
                    "pasta": f"post_{i+1:03d}",
                    "shortcode": post.shortcode,
                    "data": post_data["data_postagem"],
                    "curtidas": post_data["curtidas"],
                    "comentarios": post_data["comentarios"],
                    "tipo": post_data["tipo"],
                    "num_midias": len(post_data["midia"]["multiplas_midias"]),
                    "hashtags": post_data["hashtags"],
                    "arquivo_imagem": f"post_{i+1:03d}_{post.shortcode}.jpg",
                    "arquivo_json": f"post_{i+1:03d}_{post.shortcode}.json"
                })
                
                posts_baixados += 1
                
                # Pequena pausa entre posts
                time.sleep(2)
            
            # Salva índice geral
            indice_path = os.path.join(pasta_principal, "indice_posts.json")
            with open(indice_path, 'w', encoding='utf-8') as f:
                json.dump({
                    "perfil": profile_data,
                    "total_posts_baixados": posts_baixados,
                    "posts": todos_posts
                }, f, ensure_ascii=False, indent=2)
            
            print(f"\n{'='*60}")
            print(f"✅ Download concluído com sucesso!")
            print(f"   📁 Pasta principal: {pasta_principal}")
            print(f"   📊 Posts baixados: {posts_baixados}")
            print(f"   📂 Pastas criadas:")
            
            # Lista as pastas criadas
            for post in todos_posts[:5]:  # Mostra os primeiros 5
                print(f"      - {post['pasta']}/")
            if len(todos_posts) > 5:
                print(f"      - ... e mais {len(todos_posts)-5} pastas")
            
            return {
                "status": "sucesso",
                "pasta": pasta_principal,
                "posts_baixados": posts_baixados,
                "perfil": profile_data
            }
            
        except instaloader.exceptions.ProfileNotExistsException:
            print(f"❌ Perfil '{username}' não encontrado!")
            return {"status": "erro", "motivo": "perfil_nao_encontrado"}
        except Exception as e:
            print(f"❌ Erro inesperado: {e}")
            import traceback
            traceback.print_exc()
            return {"status": "erro", "motivo": str(e)}

In [3]:
perfil = "edelkoortnyc"  # Exemplo: "natgeo", "nasa", etc.
destino = "C:/Users/Kelvin/Desktop/IC/scrapping/imagens/instagram"

downloader = InstagramPostDownloader(pasta_base=destino)

# Opcional: fazer login (recomendado)
# downloader.fazer_login("seu_username", "sua_senha")

resultado = downloader.baixar_posts(
        username=perfil,
        num_posts=20,
        incluir_comentarios=False  # Mude para True se quiser comentários (mais lento)
    )
    
if resultado["status"] == "sucesso":
    print(f"\n🔍 Para ver os dados, abra a pasta: {resultado['pasta']}")


📸 Iniciando download do perfil: @edelkoortnyc

📊 Perfil: 
   Seguidores: 11,560
   Posts totais: 362
   Baixando últimos 20 posts...

📥 Post 1/20 (ID: DVwDsxwkvsb)

📥 Post 2/20 (ID: DVtUdjSFBYB)

📥 Post 3/20 (ID: DVhBiDbkm3G)

📥 Post 4/20 (ID: DVba41FEj7P)

📥 Post 5/20 (ID: DVRARA_jIf-)

📥 Post 6/20 (ID: DVJ5hUEIEbJ)

📥 Post 7/20 (ID: DU-62PxDBH4)

📥 Post 8/20 (ID: DU5pCujDJF8)

📥 Post 9/20 (ID: DUshZE6j-P1)

📥 Post 10/20 (ID: DUoaDsAFI2Y)

📥 Post 11/20 (ID: DUizLRHj9Qy)

📥 Post 12/20 (ID: DUa5mDKjxnb)


JSON Query to graphql/query: 403 Forbidden when accessing https://www.instagram.com/graphql/query [retrying; skip with ^C]



📥 Post 13/20 (ID: DUYh2eyD6a7)

📥 Post 14/20 (ID: DUTMrr1E5mO)


JSON Query to graphql/query: 403 Forbidden when accessing https://www.instagram.com/graphql/query [retrying; skip with ^C]



📥 Post 15/20 (ID: DTyGDSAExHC)

📥 Post 16/20 (ID: DTh3-q2kiFr)

📥 Post 17/20 (ID: DTdwCasjyIi)

📥 Post 18/20 (ID: DTS5PvuDy_I)

📥 Post 19/20 (ID: DS-7Bp2Dzbx)

📥 Post 20/20 (ID: DSvHTRelCDQ)

✅ Download concluído com sucesso!
   📁 Pasta principal: C:/Users/Kelvin/Desktop/IC/scrapping/imagens/instagram\edelkoortnyc_20260314_203055
   📊 Posts baixados: 20
   📂 Pastas criadas:
      - post_001/
      - post_002/
      - post_003/
      - post_004/
      - post_005/
      - ... e mais 15 pastas

🔍 Para ver os dados, abra a pasta: C:/Users/Kelvin/Desktop/IC/scrapping/imagens/instagram\edelkoortnyc_20260314_203055
